# 1. Connect to a postgreSQL database using *psycopg* with login data stored in an environment variable.

In [2]:
import os
import psycopg2

DATABASE_URL = os.getenv("DATABASE_URL")

conn = psycopg2.connect(DATABASE_URL)

print(conn)

conn.close()

<connection object at 0x00000149163F3780; dsn: 'user=postgres password=xxx dbname=exercises host=localhost port=5432', closed: 0>


# 2. Create a table called *workers* with fields *name*, *surname*, *position*, *hire date* and *salary*. Check if the table already exists. Insert a constraint for rows to be unique with name and surname. 

In [4]:
import os
import psycopg2

DATABASE_URL = os.getenv("DATABASE_URL")

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

create_table_sql = """
CREATE TABLE IF NOT EXISTS workers (
    worker_id BIGSERIAL PRIMARY KEY,
    name VARCHAR(25),
    surname VARCHAR(50),
    position VARCHAR(150),
    hire_date DATE,
    salary NUMERIC,
    CONSTRAINT uq_worker_name UNIQUE (name, surname)
);
"""

cur.execute(create_table_sql)
conn.commit()

cur.execute("SELECT * FROM workers;")

rows = cur.fetchall()

print("connected")
print(len(rows))

for row in rows:
    print(row)

cur.close()
conn.close()

connected
0


# 3. Add 5 rows to the table *workers*.

In [5]:
import os
import psycopg2

DATABASE_URL = os.getenv("DATABASE_URL")

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

data = [
    ('Jaden', 'Clark', 'Software Engineer', '2012-09-30', 65000),
    ('Magnus', 'Kampinsky', 'Software Architect', '1996-07-21', 82000),
    ('Olivia', 'Turner', 'Data Scientist', '2015-03-14', 71000),
    ('Ethan', 'Nguyen', 'Backend Developer', '2018-11-02', 68000),
    ('Sophia', 'Martinez', 'DevOps Engineer', '2013-06-25', 75000)
]

insert_sql = """
INSERT INTO workers (name, surname, position, hire_date, salary)
VALUES (%s, %s, %s, %s, %s)
ON CONFLICT (name, surname) DO NOTHING;
"""

cur.executemany(insert_sql, data)
conn.commit()

cur.execute("SELECT * FROM workers;")
rows = cur.fetchall()

for row in rows:
    print(row)

cur.close()
conn.close()

(1, 'Jaden', 'Clark', 'Software Engineer', datetime.date(2012, 9, 30), Decimal('65000'))
(2, 'Magnus', 'Kampinsky', 'Software Architect', datetime.date(1996, 7, 21), Decimal('82000'))
(3, 'Olivia', 'Turner', 'Data Scientist', datetime.date(2015, 3, 14), Decimal('71000'))
(4, 'Ethan', 'Nguyen', 'Backend Developer', datetime.date(2018, 11, 2), Decimal('68000'))
(5, 'Sophia', 'Martinez', 'DevOps Engineer', datetime.date(2013, 6, 25), Decimal('75000'))


# 4. Create another table called projects_assignment, consisting of columns *project_id*, *project_name*, *assignee_id* and *assignment_time_ratio*.  Make assignee_id to be a foreign key referencing the *worker_id* from *workers*. Fill the table with distinct projects .

In [21]:
import os
import psycopg2

DATABASE_URL = os.getenv("DATABASE_URL")

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

create_table_sql = """
CREATE TABLE IF NOT EXISTS projects_assignment (
    project_name text,
    assignee_id BIGINT,
    assignment_time_ratio NUMERIC(4,1),

    CONSTRAINT pk_project_assignee
        PRIMARY KEY (project_name, assignee_id),

    CONSTRAINT fk_assignee
        FOREIGN KEY (assignee_id)
        REFERENCES workers(worker_id),

    CONSTRAINT uq_project_assignee
        UNIQUE (project_name, assignee_id)
);
"""

cur.execute(create_table_sql)
conn.commit()

data = [
    ('Platform Front-End', 1, 100.0),
    ('Platform Front-End', 2, 50.0),
    ('Platform Front-End', 5, 30.0),
    ('Platform Back-End', 4, 100.0),
    ('Platform Back-End', 2, 50.0),
    ('Platform Back-End', 5, 30.0),
    ('Platform ML Analytics', 3, 100.0),
    ('Platform ML Analytics', 5, 40.0)
]

insert_sql = """
INSERT INTO projects_assignment (project_name, assignee_id, assignment_time_ratio)
VALUES (%s, %s, %s)
ON CONFLICT (project_name, assignee_id) DO NOTHING;
"""

cur.executemany(insert_sql, data)
conn.commit()

cur.execute("SELECT * FROM projects_assignment;")

rows = cur.fetchall()

print("connected")
print(len(rows))

for row in rows:
    print(row)

cur.close()
conn.close()

connected
8
('Platform Front-End', 1, Decimal('100.0'))
('Platform Front-End', 2, Decimal('50.0'))
('Platform Front-End', 5, Decimal('30.0'))
('Platform Back-End', 4, Decimal('100.0'))
('Platform Back-End', 2, Decimal('50.0'))
('Platform Back-End', 5, Decimal('30.0'))
('Platform ML Analytics', 3, Decimal('100.0'))
('Platform ML Analytics', 5, Decimal('40.0'))


# 5. Retrieve the columns with surnames and positions lexicographically sorted by surnames from the *workers* table.

In [22]:
import os
import psycopg2

DATABASE_URL = os.getenv("DATABASE_URL")

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

cur.execute("SELECT surname, position FROM workers ORDER BY surname;")
rows = cur.fetchall()

for row in rows:
    print(row)

cur.close()
conn.close()

('Clark', 'Software Engineer')
('Kampinsky', 'Software Architect')
('Martinez', 'DevOps Engineer')
('Nguyen', 'Backend Developer')
('Turner', 'Data Scientist')


# 6. Retrieve the workers names and surnames with salaries smaller or equal 70000.

In [23]:
import os
import psycopg2

DATABASE_URL = os.getenv("DATABASE_URL")

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

cur.execute("SELECT name, surname, salary FROM workers WHERE salary <= 70000  ORDER BY salary;")
rows = cur.fetchall()

for row in rows:
    print(row)

cur.close()
conn.close()

('Jaden', 'Clark', Decimal('65000'))
('Ethan', 'Nguyen', Decimal('68000'))


# 7. Get the list of workers that do not have the word 'software' in their position description.

In [24]:
import os
import psycopg2

DATABASE_URL = os.getenv("DATABASE_URL")

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

cur.execute("SELECT name, surname, position FROM workers WHERE position NOT ILIKE '%software%';")
rows = cur.fetchall()

for row in rows:
    print(row)

cur.close()
conn.close()

('Olivia', 'Turner', 'Data Scientist')
('Ethan', 'Nguyen', 'Backend Developer')
('Sophia', 'Martinez', 'DevOps Engineer')


# 8. Show the table *project_assignments*, but with *name*, *surname* and *position* instead of just *assignee_id*

In [33]:
import os
import psycopg2

DATABASE_URL = os.getenv("DATABASE_URL")

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

cur.execute("""
    SELECT projects_assignment.project_name, workers.name, workers.surname, workers.position, projects_assignment.assignment_time_ratio 
    FROM projects_assignment JOIN workers
    ON projects_assignment.assignee_id = workers.worker_id
    ORDER BY projects_assignment.project_name;
    """)
rows = cur.fetchall()

for row in rows:
    print(row)

cur.close()
conn.close()

('Platform Back-End', 'Ethan', 'Nguyen', 'Backend Developer', Decimal('100.0'))
('Platform Back-End', 'Magnus', 'Kampinsky', 'Software Architect', Decimal('50.0'))
('Platform Back-End', 'Sophia', 'Martinez', 'DevOps Engineer', Decimal('30.0'))
('Platform Front-End', 'Jaden', 'Clark', 'Software Engineer', Decimal('100.0'))
('Platform Front-End', 'Magnus', 'Kampinsky', 'Software Architect', Decimal('50.0'))
('Platform Front-End', 'Sophia', 'Martinez', 'DevOps Engineer', Decimal('30.0'))
('Platform ML Analytics', 'Olivia', 'Turner', 'Data Scientist', Decimal('100.0'))
('Platform ML Analytics', 'Sophia', 'Martinez', 'DevOps Engineer', Decimal('40.0'))


# 9. Ensure that the column with *assignment_time_ratio* in *projects_assignment* sums to around $100.0 \% $ for each worker across projects.

In [32]:
import os
import psycopg2

DATABASE_URL = os.getenv("DATABASE_URL")

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

cur.execute("""
    SELECT 
    workers.name, 
    workers.surname, 
    SUM(projects_assignment.assignment_time_ratio) AS check_sum
    FROM projects_assignment
    JOIN workers
        ON projects_assignment.assignee_id = workers.worker_id
    GROUP BY 
        workers.name, 
        workers.surname
    ORDER BY workers.name;
    """)
rows = cur.fetchall()

for row in rows:
    print(row)

cur.close()
conn.close()

('Ethan', 'Nguyen', Decimal('100.0'))
('Jaden', 'Clark', Decimal('100.0'))
('Magnus', 'Kampinsky', Decimal('100.0'))
('Olivia', 'Turner', Decimal('100.0'))
('Sophia', 'Martinez', Decimal('100.0'))


# 10. Create another table with columns *worker_id* as a foreign key referencing the *worker_id* from the *workers* table and four columns representing number of total working hours per week, denoted by *hours_week_1*,...,*hours_week_2* and fill it with some dummy numbers for each worker.

In [72]:
import os
import psycopg2

DATABASE_URL = os.getenv("DATABASE_URL")

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

create_table_sql = """
CREATE TABLE IF NOT EXISTS working_hours_per_week (
    worker_id BIGINT,
    week_1_minutes BIGINT,
    week_2_minutes BIGINT,
    week_3_minutes BIGINT,
    week_4_minutes BIGINT,

    CONSTRAINT fk_assignee
        FOREIGN KEY (worker_id)
        REFERENCES workers(worker_id)

);
"""

cur.execute(create_table_sql)
conn.commit()

data = [
    (1, 2285, 2140, 2712, 2395),
    (2, 2418, 1996, 2854, 2331),
    (3, 2197, 2235, 2620, 2458),
    (4, 2369, 2108, 2789, 2294),
    (5, 2451, 2267, 2558, 2419)
]

insert_sql = """
INSERT INTO working_hours_per_week (worker_id, week_1_minutes, week_2_minutes, week_3_minutes, week_4_minutes)
VALUES (%s, %s, %s, %s, %s )

"""

cur.executemany(insert_sql, data)
conn.commit()

cur.execute("SELECT * FROM working_hours_per_week;")

rows = cur.fetchall()

print("connected")
print(len(rows))

for row in rows:
    print(row)

cur.close()
conn.close()

connected
5
(1, 2285, 2140, 2712, 2395)
(2, 2418, 1996, 2854, 2331)
(3, 2197, 2235, 2620, 2458)
(4, 2369, 2108, 2789, 2294)
(5, 2451, 2267, 2558, 2419)


# 11. Show the table *working_hours_per_week* in an unstacked (long table) form using *UNION ALL* by adding columns *week_* and *value*.

In [73]:
import os
import psycopg2

DATABASE_URL = os.getenv("DATABASE_URL")

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

cur.execute("""
    SELECT worker_id, 'week_1' AS week, week_1_minutes AS minutes
    FROM working_hours_per_week
    
    UNION ALL
    
    SELECT worker_id, 'week_2' AS week, week_2_minutes AS minutes
    FROM working_hours_per_week
    
    UNION ALL
    
    SELECT worker_id, 'week_3' AS week, week_3_minutes AS minutes
    FROM working_hours_per_week
    
    UNION ALL
    
    SELECT worker_id, 'week_4' AS week, week_4_minutes AS minutes
    FROM working_hours_per_week;
    """)
rows = cur.fetchall()

for row in rows:
    print(row)

cur.close()
conn.close()

(1, 'week_1', 2285)
(2, 'week_1', 2418)
(3, 'week_1', 2197)
(4, 'week_1', 2369)
(5, 'week_1', 2451)
(1, 'week_2', 2140)
(2, 'week_2', 1996)
(3, 'week_2', 2235)
(4, 'week_2', 2108)
(5, 'week_2', 2267)
(1, 'week_3', 2712)
(2, 'week_3', 2854)
(3, 'week_3', 2620)
(4, 'week_3', 2789)
(5, 'week_3', 2558)
(1, 'week_4', 2395)
(2, 'week_4', 2331)
(3, 'week_4', 2458)
(4, 'week_4', 2294)
(5, 'week_4', 2419)


# 12. Show the table *working_hours_per_week* in an unstacked (long table) form using *CROSS JOIN LATERAL* by adding columns *week_* and *value*.

In [74]:
import os
import psycopg2

DATABASE_URL = os.getenv("DATABASE_URL")

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

cur.execute("""
    SELECT 
    worker_id,
    week_label,
    minutes
    FROM working_hours_per_week
    CROSS JOIN LATERAL (
        VALUES
            ('week_1', week_1_minutes),
            ('week_2', week_2_minutes),
            ('week_3', week_3_minutes),
            ('week_4', week_4_minutes)
    ) AS v(week_label, minutes);
    """)
rows = cur.fetchall()

for row in rows:
    print(row)

cur.close()
conn.close()

(1, 'week_1', 2285)
(1, 'week_2', 2140)
(1, 'week_3', 2712)
(1, 'week_4', 2395)
(2, 'week_1', 2418)
(2, 'week_2', 1996)
(2, 'week_3', 2854)
(2, 'week_4', 2331)
(3, 'week_1', 2197)
(3, 'week_2', 2235)
(3, 'week_3', 2620)
(3, 'week_4', 2458)
(4, 'week_1', 2369)
(4, 'week_2', 2108)
(4, 'week_3', 2789)
(4, 'week_4', 2294)
(5, 'week_1', 2451)
(5, 'week_2', 2267)
(5, 'week_3', 2558)
(5, 'week_4', 2419)


# 13. Save the long version of *working_hours_per_week* into a CSV file.

In [75]:
import psycopg2
import csv
import os

conn = psycopg2.connect(os.getenv("DATABASE_URL"))
cur = conn.cursor()

cur.execute("""
    SELECT 
        w.worker_id,
        v.week_label,
        v.minutes
    FROM working_hours_per_week w
    CROSS JOIN LATERAL (
        VALUES
            ('week_1', w.week_1_minutes),
            ('week_2', w.week_2_minutes),
            ('week_3', w.week_3_minutes),
            ('week_4', w.week_4_minutes)
    ) AS v(week_label, minutes);
""")

rows = cur.fetchall()

with open("working_hours_long.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["worker_id", "week_label", "minutes"])
    writer.writerows(rows)

cur.close()
conn.close()

# 14. Create a new table corresponding to the long version of *working_hours_per_week*, name it *working_hours_per_week_long* and load the saved CSV file into it.

In [86]:
import psycopg2
import os

conn = psycopg2.connect(os.getenv("DATABASE_URL"))
cur = conn.cursor()

cur.execute("""
    CREATE TABLE IF NOT EXISTS working_hours_long (
        worker_id BIGINT,
        week_label TEXT,
        minutes BIGINT,

        CONSTRAINT fk_worker
            FOREIGN KEY (worker_id)
            REFERENCES workers(worker_id)
    );
""")

with open("working_hours_long.csv", "r") as f:
    cur.copy_expert("""
        COPY working_hours_long (worker_id, week_label, minutes)
        FROM STDIN
        WITH (FORMAT CSV, HEADER TRUE)
    """, f)

conn.commit()

cur.execute("SELECT * FROM working_hours_long;")
rows = cur.fetchall()

for row in rows:
    print(row)

cur.close()
conn.close()

(1, 'week_1', 2285)
(1, 'week_2', 2140)
(1, 'week_3', 2712)
(1, 'week_4', 2395)
(2, 'week_1', 2418)
(2, 'week_2', 1996)
(2, 'week_3', 2854)
(2, 'week_4', 2331)
(3, 'week_1', 2197)
(3, 'week_2', 2235)
(3, 'week_3', 2620)
(3, 'week_4', 2458)
(4, 'week_1', 2369)
(4, 'week_2', 2108)
(4, 'week_3', 2789)
(4, 'week_4', 2294)
(5, 'week_1', 2451)
(5, 'week_2', 2267)
(5, 'week_3', 2558)
(5, 'week_4', 2419)


# 15. Show JOIN of the table *working_hours_per_week_long* with *projects_assignment* showing *name*, *surname*, *week_label*, *minutes*, *assignment_time_ratio*. Save it into a new table called worker_project_view.

In [94]:
import os
import psycopg2

DATABASE_URL = os.getenv("DATABASE_URL")

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

cur.execute("""
    CREATE TABLE worker_project_view AS
    SELECT 
    workers.name, 
    workers.surname, 
    projects_assignment.project_name,
    working_hours_long.week_label,
    working_hours_long.minutes,
    projects_assignment.assignment_time_ratio
    FROM working_hours_long
    JOIN workers
        ON working_hours_long.worker_id = workers.worker_id
    JOIN projects_assignment
        ON working_hours_long.worker_id = projects_assignment.assignee_id
    GROUP BY 
        workers.name, 
        workers.surname,
        projects_assignment.project_name,
        working_hours_long.week_label,
        working_hours_long.minutes,
        projects_assignment.assignment_time_ratio
    ORDER BY workers.name;
    """)

conn.commit()

cur.execute("SELECT * FROM worker_project_view;")
rows = cur.fetchall()

for row in rows:
    print(row)

cur.close()
conn.close()

('Ethan', 'Nguyen', 'Platform Back-End', 'week_1', 2369, Decimal('100.0'))
('Ethan', 'Nguyen', 'Platform Back-End', 'week_2', 2108, Decimal('100.0'))
('Ethan', 'Nguyen', 'Platform Back-End', 'week_3', 2789, Decimal('100.0'))
('Ethan', 'Nguyen', 'Platform Back-End', 'week_4', 2294, Decimal('100.0'))
('Jaden', 'Clark', 'Platform Front-End', 'week_1', 2285, Decimal('100.0'))
('Jaden', 'Clark', 'Platform Front-End', 'week_2', 2140, Decimal('100.0'))
('Jaden', 'Clark', 'Platform Front-End', 'week_3', 2712, Decimal('100.0'))
('Jaden', 'Clark', 'Platform Front-End', 'week_4', 2395, Decimal('100.0'))
('Magnus', 'Kampinsky', 'Platform Back-End', 'week_1', 2418, Decimal('50.0'))
('Magnus', 'Kampinsky', 'Platform Back-End', 'week_2', 1996, Decimal('50.0'))
('Magnus', 'Kampinsky', 'Platform Back-End', 'week_3', 2854, Decimal('50.0'))
('Magnus', 'Kampinsky', 'Platform Back-End', 'week_4', 2331, Decimal('50.0'))
('Magnus', 'Kampinsky', 'Platform Front-End', 'week_1', 2418, Decimal('50.0'))
('Magnus'

# 16. Add new column called *minutes_per_week_per_project* to the table *worker_project_view* and fill it with corresponding values calculated with values from the same table in corresponding columns.

In [98]:
import os
import psycopg2

DATABASE_URL = os.getenv("DATABASE_URL")

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

cur.execute("""
    ALTER TABLE worker_project_view
    ADD COLUMN minutes_per_week_per_project NUMERIC;
    """)

cur.execute("""
    UPDATE worker_project_view
    SET minutes_per_week_per_project =
        CEIL(minutes * (assignment_time_ratio / 100.0));
    """)

conn.commit()

cur.execute("SELECT * FROM worker_project_view;")
rows = cur.fetchall()

for row in rows:
    print(row)

cur.close()
conn.close()

('Ethan', 'Nguyen', 'Platform Back-End', 'week_1', 2369, Decimal('100.0'), Decimal('2369'))
('Ethan', 'Nguyen', 'Platform Back-End', 'week_2', 2108, Decimal('100.0'), Decimal('2108'))
('Ethan', 'Nguyen', 'Platform Back-End', 'week_3', 2789, Decimal('100.0'), Decimal('2789'))
('Ethan', 'Nguyen', 'Platform Back-End', 'week_4', 2294, Decimal('100.0'), Decimal('2294'))
('Jaden', 'Clark', 'Platform Front-End', 'week_1', 2285, Decimal('100.0'), Decimal('2285'))
('Jaden', 'Clark', 'Platform Front-End', 'week_2', 2140, Decimal('100.0'), Decimal('2140'))
('Jaden', 'Clark', 'Platform Front-End', 'week_3', 2712, Decimal('100.0'), Decimal('2712'))
('Jaden', 'Clark', 'Platform Front-End', 'week_4', 2395, Decimal('100.0'), Decimal('2395'))
('Magnus', 'Kampinsky', 'Platform Back-End', 'week_1', 2418, Decimal('50.0'), Decimal('1209'))
('Magnus', 'Kampinsky', 'Platform Back-End', 'week_2', 1996, Decimal('50.0'), Decimal('998'))
('Magnus', 'Kampinsky', 'Platform Back-End', 'week_3', 2854, Decimal('50.0'

# 17. Delete columns *minutes* and *assignment_time_ratio* from the table *worker_project_view*.

In [99]:
import os
import psycopg2

DATABASE_URL = os.getenv("DATABASE_URL")

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

cur.execute("""
    ALTER TABLE worker_project_view
    DROP COLUMN minutes,
    DROP COLUMN assignment_time_ratio;
    """)

conn.commit()

cur.execute("SELECT * FROM worker_project_view;")
rows = cur.fetchall()

for row in rows:
    print(row)

cur.close()
conn.close()

('Ethan', 'Nguyen', 'Platform Back-End', 'week_1', Decimal('2369'))
('Ethan', 'Nguyen', 'Platform Back-End', 'week_2', Decimal('2108'))
('Ethan', 'Nguyen', 'Platform Back-End', 'week_3', Decimal('2789'))
('Ethan', 'Nguyen', 'Platform Back-End', 'week_4', Decimal('2294'))
('Jaden', 'Clark', 'Platform Front-End', 'week_1', Decimal('2285'))
('Jaden', 'Clark', 'Platform Front-End', 'week_2', Decimal('2140'))
('Jaden', 'Clark', 'Platform Front-End', 'week_3', Decimal('2712'))
('Jaden', 'Clark', 'Platform Front-End', 'week_4', Decimal('2395'))
('Magnus', 'Kampinsky', 'Platform Back-End', 'week_1', Decimal('1209'))
('Magnus', 'Kampinsky', 'Platform Back-End', 'week_2', Decimal('998'))
('Magnus', 'Kampinsky', 'Platform Back-End', 'week_3', Decimal('1427'))
('Magnus', 'Kampinsky', 'Platform Back-End', 'week_4', Decimal('1166'))
('Magnus', 'Kampinsky', 'Platform Front-End', 'week_1', Decimal('1209'))
('Magnus', 'Kampinsky', 'Platform Front-End', 'week_2', Decimal('998'))
('Magnus', 'Kampinsky', 